# Homework Option B: Build a Diffusion Generator for a Topic of Your Choice

Time: up to 60 minutes. Runs on CPU. No GPU, Drive, or API key needed.

This notebook is a blank canvas. Pick any small numeric-vector topic you like
(colors, melodies, shapes, daily patterns, or your own idea) and build a
diffusion model for it, using the same math from the workshop.
Look for `FILL_IN` and `TODO` markers.
See `Homework_OptionB_ChooseYourDiffusion.md` for full instructions.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
print('Setup ready.')

---

## Step 1 and 2: Choose Your Topic and Build Your Dataset

Write code that produces a tensor `X` of shape `[n_samples, dim]`.
Some starter examples are given below as comments. Uncomment one,
write your own, or invent something new.

In [ ]:
# TODO: define your dataset here. X should be a tensor of shape [n_samples, dim]
# Aim for at least 200 examples and roughly normalize values to [-1, 1].

# --- Example A: Colors (dim=3) ---
# def hex_to_rgb_tensor(hex_list):
#     rgb = []
#     for h in hex_list:
#         h = h.lstrip('#')
#         rgb.append([int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)])
#     rgb = torch.tensor(rgb, dtype=torch.float32) / 255.0
#     return rgb * 2 - 1
# HEX_COLORS = ['#FFD1DC', '#FFE4B5', '#D1FFD1', '#C1F0F0', '#D1D1FF']  # add more for variety
# X = hex_to_rgb_tensor(HEX_COLORS)

# --- Example B: Simple 2D shape (dim=2) ---
# from sklearn.datasets import make_circles
# X, _ = make_circles(n_samples=500, noise=0.05, factor=0.5)
# X = torch.tensor(X, dtype=torch.float32)
# X = (X - X.mean(0)) / X.std(0) * 0.5

# --- Example C: Short melody (dim=4), pitches as MIDI numbers ---
# Generates random 4-note melodies centered around a chosen scale
# base_notes = torch.tensor([60, 64, 67, 72], dtype=torch.float32)  # C major chord notes
# X = base_notes + torch.randn(500, 4) * 1.5
# X = (X - X.mean()) / X.std() * 0.5   # normalize roughly to [-1, 1]

X = FILL_IN   # TODO: replace with your dataset
DIM = X.shape[1]
print(f'Dataset shape: {X.shape}')

---

## Step 3: Noise Schedule

This part does not depend on what your data represents.
It works the same way no matter how many numbers are in each example.

In [ ]:
T = 100
B = torch.linspace(1e-4, 0.02, T)
a = 1 - B
a_bar = torch.cumprod(a, dim=0)
sqrt_a_bar = torch.sqrt(a_bar)
sqrt_one_minus_a_bar = torch.sqrt(1 - a_bar)
sqrt_a_inv = torch.sqrt(1 / a)
pred_noise_coeff = (1 - a) / sqrt_one_minus_a_bar

print('Noise schedule ready.')

### FILL_IN 1: Closed-Form Forward Diffusion

Same formula as Part 1, Exercise 2 in the workshop.

In [ ]:
def q(x_0, t):
    noise = torch.randn_like(x_0)
    sqrt_a_bar_t    = sqrt_a_bar[t][:, None]
    sqrt_1m_a_bar_t = sqrt_one_minus_a_bar[t][:, None]
    # FILL_IN: combine x_0 and noise using the two coefficients above
    x_t = FILL_IN
    return x_t, noise

<details>
<summary><b>Click to show the solution</b></summary>

```python
x_t = sqrt_a_bar_t * x_0 + sqrt_1m_a_bar_t * noise
```
</details>

---

## Step 4: The Model and Training

A tiny MLP that works for any `DIM`. No changes needed here.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, t):
        t_norm = (t.float() / T).unsqueeze(1)
        inp = torch.cat([x, t_norm], dim=1)
        return self.net(inp)

model = TinyMLP(dim=DIM)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-3)
STEPS = 3000
BATCH = min(64, len(X))

for step in range(STEPS):
    optimizer.zero_grad()
    idx = torch.randint(0, len(X), (BATCH,))
    x0  = X[idx]
    t   = torch.randint(0, T, (BATCH,))
    x_noisy, noise = q(x0, t)
    pred = model(x_noisy, t)
    loss = F.mse_loss(pred, noise)
    loss.backward()
    optimizer.step()
    if step % 500 == 0:
        print(f'Step {step}/{STEPS} | loss: {loss.item():.4f}')

print('Training done.')

### FILL_IN 2: Reverse Diffusion

Same formula as Part 2, Exercise 1 in the workshop.

In [ ]:
@torch.no_grad()
def reverse_q(x_t, t, e_t):
    coeff        = pred_noise_coeff[t][:, None]
    sqrt_a_inv_t = sqrt_a_inv[t][:, None]
    # FILL_IN: combine x_t, e_t, coeff, and sqrt_a_inv_t
    u_t = FILL_IN
    if t[0] == 0:
        return u_t
    B_prev = B[t - 1][:, None]
    return u_t + torch.sqrt(B_prev) * torch.randn_like(x_t)

<details>
<summary><b>Click to show the solution</b></summary>

```python
u_t = sqrt_a_inv_t * (x_t - coeff * e_t)
```
</details>

---

## Step 5: Generate and Visualize

Write a visualization function appropriate for your topic.
A few starter snippets are given as comments below.

In [ ]:
@torch.no_grad()
def sample(n=16):
    x = torch.randn(n, DIM)
    for i in range(T - 1, -1, -1):
        t = torch.full((n,), i)
        e_t = model(x, t)
        x = reverse_q(x, t, e_t)
    return x.clamp(-1, 1)

generated = sample(n=16)
print(f'Generated shape: {generated.shape}')

# TODO: write your own visualization

# --- If your topic is colors (dim=3) ---
# colors_disp = ((generated + 1) / 2).clamp(0, 1).numpy()
# fig, ax = plt.subplots(figsize=(len(generated), 1.5))
# for i, c in enumerate(colors_disp):
#     ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
# ax.set_xlim(0, len(generated)); ax.set_ylim(0, 1); ax.axis('off')
# plt.show()

# --- If your topic is 2D shapes (dim=2) ---
# plt.scatter(X[:,0], X[:,1], s=5, alpha=0.3, label='real')
# plt.scatter(generated[:,0], generated[:,1], s=5, alpha=0.3, label='generated')
# plt.legend(); plt.axis('equal'); plt.show()

# --- If your topic is melodies or sequences (any dim) ---
# fig, axes = plt.subplots(4, 4, figsize=(10, 6))
# for ax, melody in zip(axes.flat, generated):
#     ax.bar(range(DIM), melody.numpy())
#     ax.set_ylim(-1, 1)
# plt.tight_layout(); plt.show()

---

## Report

**1. What topic did you choose, and why?**

_(your answer here)_

**2. Did the generated samples look similar in style to your training data? Give one example that worked well and one that did not.**

_(your answer here)_

**3. If you had more time, what would you change to improve the results?**

_(your answer here)_